# WP06: LangGraph Agent Core — Prototype Test

The agent is responsible for autonomously planning a full 
milonga session — deciding which tandas (groups of tango tracks) to play, 
selecting cortinas (short breaks between tandas), and adapting to feedback.

## What we are testing
We run a minimal end-to-end session to verify:

1. **State initialization** — the agent correctly sets up a `MilongaSession` 
   and computes an energy arc (the target energy curve across the session)

2. **Tanda planning loop** — the `tanda_planner` node uses a Gemini LLM with 
   tools to search the catalog and plan each tanda slot

3. **Cortina selection** — after each tanda, the `cortina_selector` node picks 
   a short non-tango break from the cortina pool

4. **Queue publishing** — the `queue_publisher` node advances the session index 
   and emits each tanda to the queue

5. **Graph routing** — the conditional edges correctly loop through all tandas 
   and terminate cleanly at `session_summary`

## What is stubbed / not yet connected
- Cortina selection returns a default placeholder (real pool comes from WP09)
- No UI connection yet (Streamlit integration comes in WP10)
- No RAG/ChromaDB yet (comes from WP07)
- Tanda validator stubs will be replaced by real WP05 logic

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from atdj.agent.graph import build_graph
from atdj.schemas.session import MilongaSession
from datetime import datetime
import uuid

session = MilongaSession(
    id=str(uuid.uuid4()),
    name="Test Session",
    started_at=datetime.now(),
    target_duration_minutes=60,
)

In [3]:
graph = build_graph()

initial_state = {
    "messages": [],
    "session": session,
    "energy_arc": [],
    "current_tanda_index": 0,
    "upcoming_tandas": [],
    "pending_feedback": [],
    "needs_cortina": False,
    "session_complete": False,
    "feedback_pending": False,
    "candidate_tracks": [],
    "current_tanda_draft": None,
    "last_agent_action": None,
    "qa_question": None,
    "qa_answer": None,
    "error_message": None,
    "retry_count": 0,
}

print("Starting agent...\n")
for step in graph.stream(initial_state):
    node_name = list(step.keys())[0]
    print(f">>> Node: {node_name}")
    state = step[node_name]
    if "messages" in state and state["messages"]:
        print(f"    {state['messages'][-1].content}")
    print()

Starting agent...

>>> Node: session_init
    Session initialized. Ready to plan tandas.

>>> Node: tanda_planner
    

>>> Node: cortina_selector
    Cortina selected: unknown

>>> Node: queue_publisher
    Tanda 1 published to queue.

>>> Node: tanda_planner
    

>>> Node: cortina_selector
    Cortina selected: unknown

>>> Node: queue_publisher
    Tanda 2 published to queue.

>>> Node: tanda_planner
    

>>> Node: cortina_selector
    Cortina selected: unknown

>>> Node: queue_publisher
    Tanda 3 published to queue.

>>> Node: tanda_planner
    

>>> Node: cortina_selector
    Cortina selected: unknown

>>> Node: queue_publisher
    Tanda 4 published to queue.

>>> Node: session_summary
    Session complete! Planned 4 tandas.

